# What the env wheel's native payload brings, and what it costs

Runs the wheel's own self-check on one runtime and returns one JSON report: which build each
binding runs and whether it came from the payload, the light and the light SQL answers, the import
time, the payload and the installed sizes, and (on dedicated compute) the heavy SQL answers and
where the driver and the executors import GeoBrix from.

- `MODE = "serverless"`: the light checks, and two timed calls of the Unity Catalog function
  `UC_FUNCTION`, when it exists. Nothing here sets a Spark configuration or touches the JVM.
- `MODE = "dedicated"`: the light checks, then the heavy ones. `SET_NATIVE_DIR` points the JVM at the
  wheel's natives.

The notebook ends with `dbutils.notebook.exit(<report>)`; read it with `jobs get-run-output`.


In [ ]:
import json
import os
import time

# "serverless" or "dedicated". Set it before this cell (the pip notebook does) or with --set-var.
MODE = globals().get("MODE", "serverless")
SET_NATIVE_DIR = True  # dedicated only: point the JVM at the wheel's natives
SCRATCH = "/Volumes/geospatial_docs/geobrix/sample-data/_probe/natives"
UC_FUNCTION = "geospatial_docs.gbx_zarr_scratch.gbx_probe_native_payload"


In [ ]:
t0 = time.perf_counter()
import databricks.labs.gbx  # noqa: E402,F401

import_s = round(time.perf_counter() - t0, 3)

from databricks.labs.gbx._native import _selfcheck  # noqa: E402

report = {
    "mode": MODE,
    "import_s": import_s,
    "stack": _selfcheck.stack(),
    "light": _selfcheck.light(),
    "light_sql": _selfcheck.light_sql(spark),
    "payload_size_mb": _selfcheck.payload_size_mb(),
    "installed_closure_mb": _selfcheck.installed_closure_mb(),
}
print(json.dumps({k: report[k] for k in ("mode", "import_s", "payload_size_mb", "installed_closure_mb")}))


In [ ]:
if MODE == "dedicated":
    from databricks.labs.gbx import _native
    from databricks.labs.gbx.manager.jvm import NATIVE_DIR_KEY

    os.makedirs(SCRATCH, exist_ok=True)
    if SET_NATIVE_DIR:  # classic compute only: Spark Connect has no JVM configuration to set
        spark.conf.set(NATIVE_DIR_KEY, _native.native_dir())
    try:
        report["heavy"] = _selfcheck.heavy_sql(spark, SCRATCH)
    except Exception as exc:  # recorded: the report is the result
        report["heavy"] = {"ok": False, "error": f"{type(exc).__name__}: {str(exc)[:500]}"}
    report["paths"] = _selfcheck.install_paths(spark)

In [ ]:
if MODE == "serverless":
    uc = {"function": UC_FUNCTION}
    try:
        spark.sql(f"DESCRIBE FUNCTION {UC_FUNCTION}").collect()
        uc["exists"] = True
    except Exception:
        uc["exists"] = False
    if uc["exists"]:
        for i in (1, 2):
            t = time.perf_counter()
            try:
                uc[f"call{i}"] = json.loads(spark.sql(f"SELECT {UC_FUNCTION}({i}) AS r").first().r)
            except Exception as exc:
                uc[f"call{i}"] = {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}
            uc[f"call{i}_s"] = round(time.perf_counter() - t, 2)
    report["uc"] = uc
    print(json.dumps(uc))


In [ ]:
ok = report["light"]["ok"] and report["light_sql"]["ok"] and report.get("heavy", {"ok": True})["ok"]
dbutils.notebook.exit(json.dumps({"ok": ok, **report}, default=str))  # read with jobs get-run-output
